# Refresh / Content Opportunity Scoring — Starter EDA
This notebook connects to the FlyRank internship warehouse via DuckDB and Hugging Face, explores the key tables for a refresh/opportunity lane, and builds a safe content-level feature table.

## What this notebook does
1. Installs the required DuckDB and Hugging Face packages.
2. Prompts for a Hugging Face read token safely.
3. Connects to `hf://datasets/FlyRank/internship-warehouse`.
4. Lists the relevant tables and sample columns.
5. Builds a content-level feature table for refresh/opportunity scoring.
6. Saves a small safe sample to `work/outputs` for downstream modeling.

In [1]:
# Install required packages if they are not already available.
# In Colab, use: %pip install -q duckdb huggingface_hub
import importlib.util
import subprocess
import sys

requirements = ["duckdb", "huggingface_hub"]
for pkg in requirements:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)

In [ ]:
import os
import getpass
from pathlib import Path

import duckdb
import pandas as pd

# If the notebook is running in Colab, the token can also come from the HF_TOKEN secret.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata

        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN.startswith("hf_"), "Enter a valid Hugging Face read token starting with hf_."

ROOT = Path.cwd()
# If the notebook is opened from a subfolder, climb until the repo root is found.
while not (ROOT / "work").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "work").exists(), "Could not find the repository root containing work/."

OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Connect DuckDB to the Hugging Face dataset release.
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected to FlyRank internship warehouse.")

In [ ]:
# List the available tables with row counts.
counts = []
for name, src in TABLES.items():
    try:
        n = con.sql(f"SELECT COUNT(*) AS cnt FROM {src}").fetchone()[0]
    except Exception as exc:
        n = f"error: {exc}"
    counts.append((name, n))

counts_df = pd.DataFrame(counts, columns=["table", "row_count"])
counts_df

## EDA: table schemas and sample values
Below we inspect the schema and a small sample of each table so we can choose the right fields for refresh/opportunity scoring.

In [ ]:
for name, src in TABLES.items():
    print("\n===", name, "===")
    try:
        schema = con.sql(f"DESCRIBE {src}").df()
        print(schema.head(20).to_string(index=False))
    except Exception as exc:
        print(f"Unable to describe {name}: {exc}")

In [ ]:
# Sample a few rows from each dataset.
for name, src in TABLES.items():
    print(f"\n--- Sample from {name} ---")
    try:
        sample = con.sql(f"SELECT * FROM {src} LIMIT 5").df()
        display(sample)
    except Exception as exc:
        print(f"Unable to sample {name}: {exc}")

## 1. Candidate label and safe features
For the refresh/opportunity lane, the most useful signals are:
- demand and visibility metrics (`impressions`, `clicks`, `avg_position`, `ctr`)
- trend direction / momentum (`trend_direction`, `trend_pct`, last-30d delta)
- freshness and content age (`content_age_days`, `days_since_last_update`)
- query diversity and tail exposure (`visible_queries`, `rare_impressions_share`, `anonymized_impressions_share`)

This notebook uses only pseudonymous IDs and aggregated metrics, not client names, domains, URLs, or raw search queries.

In [ ]:
# Build a content-level feature table in DuckDB SQL.
# This is a safe, aggregated view for scoring refresh opportunity.
query = f"""
WITH latest AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        ANY_VALUE(content_age_days) AS content_age_days,
        ANY_VALUE(days_since_last_update) AS days_since_last_update,
        ANY_VALUE(avg_position) AS avg_position,
        ANY_VALUE(ctr) AS ctr,
        ANY_VALUE(engagement_rate) AS engagement_rate,
        ANY_VALUE(scroll_rate) AS scroll_rate,
        ANY_VALUE(ai_traffic_pct) AS ai_traffic_pct,
        ANY_VALUE(trend_direction) AS trend_direction,
        ANY_VALUE(trend_pct) AS trend_pct,
        ANY_VALUE(word_count) AS word_count,
        ANY_VALUE(impressions_90d) AS impressions_90d,
        ANY_VALUE(clicks_90d) AS clicks_90d,
        ANY_VALUE(sessions_90d) AS sessions_90d,
        ANY_VALUE(position_tier) AS position_tier,
        ANY_VALUE(freshness_tier) AS freshness_tier
    FROM {TABLES['dim_content']}
    GROUP BY content_hash_id
),
query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_impressions_share,
        ANY_VALUE(anonymized_impressions_share) AS anonymized_impressions_share,
        MAX(impressions_90d) AS max_query_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
),
momentum AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date > MAX(report_date) OVER () - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS impressions_last_30d,
        SUM(CASE WHEN report_date <= MAX(report_date) OVER () - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS impressions_prev_30d,
        AVG(CASE WHEN report_date > MAX(report_date) OVER () - INTERVAL 30 DAY THEN gsc_avg_position END) AS avg_position_last_30d,
        AVG(CASE WHEN report_date <= MAX(report_date) OVER () - INTERVAL 30 DAY THEN gsc_avg_position END) AS avg_position_prev_30d
    FROM {TABLES['fact_daily']}
    WHERE report_date > MAX(report_date) OVER () - INTERVAL 90 DAY
    GROUP BY content_hash_id
)
SELECT
    l.content_hash_id,
    l.client_hash_id,
    l.content_age_days,
    l.days_since_last_update,
    l.avg_position,
    l.ctr,
    l.engagement_rate,
    l.scroll_rate,
    l.ai_traffic_pct,
    l.trend_direction,
    l.trend_pct,
    l.word_count,
    l.impressions_90d,
    l.clicks_90d,
    l.sessions_90d,
    l.position_tier,
    l.freshness_tier,
    q.visible_queries,
    q.rare_impressions_share,
    q.anonymized_impressions_share,
    q.max_query_impressions,
    m.impressions_last_30d,
    m.impressions_prev_30d,
    m.avg_position_last_30d,
    m.avg_position_prev_30d,
    CASE WHEN m.impressions_prev_30d > 0 THEN (m.impressions_last_30d - m.impressions_prev_30d) / m.impressions_prev_30d ELSE NULL END AS impressions_pct_change_30d,
    CASE WHEN m.avg_position_prev_30d > 0 THEN m.avg_position_prev_30d - m.avg_position_last_30d ELSE NULL END AS position_change_30d
FROM latest l
LEFT JOIN query_signals q USING (content_hash_id)
LEFT JOIN momentum m USING (content_hash_id)
LIMIT 2000
"""

feature_df = con.sql(query).df()
print("Loaded feature table with rows:", len(feature_df))
feature_df.head()

## 2. Feature sanity checks
The table above is the starting point for a refresh scoring model. It contains:
- current visibility and engagement metrics,
- freshness and content age,
- query diversity and tail exposure,
- momentum signals for last 30 days vs previous 30 days.

In [ ]:
# Basic distribution checks for key signal columns.
checks = feature_df[
    [
        "impressions_90d",
        "ctr",
        "avg_position",
        "trend_pct",
        "visible_queries",
        "anonymized_impressions_share",
        "impressions_pct_change_30d",
        "position_change_30d",
    ]
]
print(checks.describe(percentiles=[0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).T)

# Save a small safe sample for later modeling or reporting.
output_path = OUTPUT_DIR / "refresh_feature_sample.csv"
feature_df.to_csv(output_path, index=False)
print("Saved feature sample to", output_path)

## Next steps
1. Use this feature table as the basis for a transparent baseline score.
2. Build a model with a leakage-safe split (client-aware or time-aware).
3. Compare model vs baseline on the same holdout split.
4. Export the ranked action queue and turn the results into the deployed paper.

In [ ]:
<VSCode.Cell id="#VSC-fb8269b8" language="markdown">
## 3. Baseline refresh opportunity scoring
This section builds a transparent rule that uses visibility, freshness, and position to score pages that look most in need of refresh attention.
</VSCode.Cell>